# Bond Return Series Comparison

Construct correct annual holding-period returns for every available bond yield series,
so we can select the series with the most favourable expected return for the lifecycle VAR model.

**Two methods:**
- **CCV loglinear approximation** for coupon-bearing / par yields
- **Exact zero-coupon return** for GSW continuously-compounded ZC yields

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = next(
    (
        candidate for candidate in [
            Path.cwd() / "data" / "Thesisdata",
            Path.cwd().parent / "data" / "Thesisdata",
            Path.cwd() / "Thesisdata",
            Path.cwd(),
        ]
        if (candidate / "DGS1.csv").exists()
    ),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError("Could not locate Thesisdata directory with bond inputs")

DATA = str(DATA_DIR)

## 1. Load Raw Data

In [ ]:
# DGS1: 1-year Treasury yield (daily)
dgs1_raw = pd.read_csv(f'{DATA}/DGS1.csv', index_col='observation_date', parse_dates=True)
dgs1_raw['DGS1'] = pd.to_numeric(dgs1_raw['DGS1'], errors='coerce')

# AAA: Moody's AAA corporate bond yield (monthly)
aaa_raw = pd.read_csv(f'{DATA}/AAA.csv', index_col='observation_date', parse_dates=True)
aaa_raw['AAA'] = pd.to_numeric(aaa_raw['AAA'], errors='coerce')

# GS20: 20-year Treasury yield (monthly, NaN gap 1987-1992)
gs20_raw = pd.read_csv(f'{DATA}/GS20.csv', index_col='observation_date', parse_dates=True)
gs20_raw['GS20'] = pd.to_numeric(gs20_raw['GS20'], errors='coerce')

# DGS30: 30-year Treasury yield (daily)
dgs30_raw = pd.read_csv(f'{DATA}/DGS30.csv', index_col='observation_date', parse_dates=True)
dgs30_raw['DGS30'] = pd.to_numeric(dgs30_raw['DGS30'], errors='coerce')

# CPI: CPIAUCSL (monthly)
cpi_raw = pd.read_csv(f'{DATA}/CPIAUCSL.csv', index_col='observation_date', parse_dates=True)
cpi_raw['CPIAUCSL'] = pd.to_numeric(cpi_raw['CPIAUCSL'], errors='coerce')

# GSW yield curve (skip 9 header rows)
gsw = pd.read_csv(f'{DATA}/feds200628 (1).csv', skiprows=9, index_col='Date', parse_dates=True)
# Replace -999.99 and similar sentinel values with NaN
gsw = gsw.replace(-999.99, np.nan)
for col in gsw.columns:
    gsw[col] = pd.to_numeric(gsw[col], errors='coerce')

# Shiller data (for stock returns)
shiller = pd.read_excel(f'{DATA}/ie_data.xls', sheet_name='Data', header=7)
shiller = shiller.rename(columns={shiller.columns[0]: 'date_raw'})
shiller = shiller.dropna(subset=['date_raw'])
shiller = shiller[shiller['date_raw'].apply(lambda x: isinstance(x, (int, float)))]
shiller['date_raw'] = shiller['date_raw'].astype(float)
shiller['year'] = shiller['date_raw'].apply(lambda x: int(x))
shiller['month'] = shiller['date_raw'].apply(lambda x: max(1, round((x % 1) * 100)))
shiller['date'] = pd.to_datetime(shiller[['year', 'month']].assign(day=1))
shiller = shiller.set_index('date')
shiller['P_nom'] = pd.to_numeric(shiller['P'], errors='coerce')
shiller['D_nom'] = pd.to_numeric(shiller['D'], errors='coerce')

print('Data loaded.')

## 2. Resample to End-of-Year

In [ ]:
# 1-year Treasury yield (end-of-December, decimal)
y_1 = dgs1_raw['DGS1'].resample('YE-DEC').last() / 100.0
y_1.index = y_1.index.year
y_1.name = 'y_1'

# AAA yield (end-of-December, decimal)
y_aaa = aaa_raw['AAA'].resample('YE-DEC').last() / 100.0
y_aaa.index = y_aaa.index.year

# GS20 yield (end-of-December, decimal)
y_gs20 = gs20_raw['GS20'].resample('YE-DEC').last() / 100.0
y_gs20.index = y_gs20.index.year

# DGS30 yield (end-of-December, decimal)
y_dgs30 = dgs30_raw['DGS30'].resample('YE-DEC').last() / 100.0
y_dgs30.index = y_dgs30.index.year

# GSW yields (end-of-December, decimal — yields are in percent)
gsw_dec = gsw.resample('YE-DEC').last()
gsw_dec.index = gsw_dec.index.year

# CPI December values
cpi_dec = cpi_raw['CPIAUCSL'].resample('YE-DEC').last()
cpi_dec.index = cpi_dec.index.year

# Inflation: pi[T] = log(CPI_Dec_T / CPI_Dec_{T-1})
pi = np.log(cpi_dec / cpi_dec.shift(1))
pi.name = 'pi'

# Log nominal bill return: r_1[T] = log(1 + y_1[T])
r_1 = np.log(1 + y_1)

# Real bill return: rtb[T] = log(1 + y_1[T-1]) - pi[T]
rtb = r_1.shift(1) - pi
rtb.name = 'rtb'

# Stock excess return (for correlations later)
shiller['nom_ret_m'] = np.log(
    (shiller['P_nom'] + shiller['D_nom'] / 12) / shiller['P_nom'].shift(1)
)
annual_nom_stock = shiller.groupby(shiller.index.year)['nom_ret_m'].agg(['sum', 'count'])
annual_nom_stock = annual_nom_stock[annual_nom_stock['count'] == 12]['sum']
annual_nom_stock.name = 'nominal_stock'
xr = annual_nom_stock - r_1.shift(1)
xr.name = 'xr'

print(f'y_1: {y_1.dropna().index.min()}–{y_1.dropna().index.max()}')
print(f'y_aaa: {y_aaa.dropna().index.min()}–{y_aaa.dropna().index.max()}')
print(f'y_gs20: {y_gs20.dropna().index.min()}–{y_gs20.dropna().index.max()} (check for NaN gap)')
print(f'y_dgs30: {y_dgs30.dropna().index.min()}–{y_dgs30.dropna().index.max()}')
print(f'pi: {pi.dropna().index.min()}–{pi.dropna().index.max()}')

## 3. Return Construction Functions

In [ ]:
def ccv_bond_return(Y, n):
    """
    CCV loglinear approximation for log holding-period return on a 
    constant-maturity par bond portfolio.
    
    Y: coupon-equivalent yield in decimal (Series, indexed by year)
    n: bond maturity in years
    
    Returns: r_bond Series (log return realized during year T)
    """
    # Macaulay duration of par bond at yield Y
    D = (1 - (1 + Y)**(-n)) / (1 - (1 + Y)**(-1))
    y_log = np.log(1 + Y)
    # r_bond[T] = D[T-1] * log(1+Y[T-1]) - (D[T-1]-1) * log(1+Y[T])
    r_bond = D.shift(1) * y_log.shift(1) - (D.shift(1) - 1) * y_log
    return r_bond, D


def zc_bond_return(y_cc_n, y_cc_n_minus_1, n):
    """
    Exact log return on a zero-coupon bond.
    
    Buy n-year ZC at end of year T-1, sell as (n-1)-year ZC at end of year T.
    
    y_cc_n: continuously compounded ZC yield for maturity n (decimal, indexed by year)
    y_cc_n_minus_1: CC ZC yield for maturity n-1 (decimal, indexed by year)
    n: maturity in years
    
    Returns: r_zc Series
    """
    # r_zc[T] = n * y_cc_n[T-1] - (n-1) * y_cc_{n-1}[T]
    r_zc = n * y_cc_n.shift(1) - (n - 1) * y_cc_n_minus_1
    return r_zc


print('Return functions defined.')

## 4. Construct All 10 Series

In [ ]:
results = {}  # label -> dict with r_bond, xb, D, yield_series, method, n, etc.

# Helper: excess bond return
def make_xb(r_bond):
    return r_bond - r_1.shift(1)

# --- Series 1: AAA 20yr (CCV) ---
r_bond_aaa, D_aaa = ccv_bond_return(y_aaa, 20)
results['AAA 20yr'] = dict(
    r_bond=r_bond_aaa, xb=make_xb(r_bond_aaa), D=D_aaa,
    yield_series=y_aaa, method='CCV', n=20, label='AAA 20yr'
)

# --- Series 2: GS20 Treasury 20yr (CCV) ---
r_bond_gs20, D_gs20 = ccv_bond_return(y_gs20, 20)
results['GS20 20yr'] = dict(
    r_bond=r_bond_gs20, xb=make_xb(r_bond_gs20), D=D_gs20,
    yield_series=y_gs20, method='CCV', n=20, label='GS20 20yr'
)

# --- Series 3: DGS30 Treasury 30yr (CCV) ---
r_bond_dgs30, D_dgs30 = ccv_bond_return(y_dgs30, 30)
results['DGS30 30yr'] = dict(
    r_bond=r_bond_dgs30, xb=make_xb(r_bond_dgs30), D=D_dgs30,
    yield_series=y_dgs30, method='CCV', n=30, label='DGS30 30yr'
)

# --- Series 4: DGS30, assume n=20 (CCV, wrong maturity) ---
r_bond_dgs30_n20, D_dgs30_n20 = ccv_bond_return(y_dgs30, 20)
results['DGS30 n=20'] = dict(
    r_bond=r_bond_dgs30_n20, xb=make_xb(r_bond_dgs30_n20), D=D_dgs30_n20,
    yield_series=y_dgs30, method='CCV', n=20, label='DGS30 n=20'
)

# --- GSW ZC yields (decimal) ---
def get_gsw_zc(mat):
    col = f'SVENY{mat:02d}'
    return gsw_dec[col] / 100.0

def get_gsw_par(mat):
    col = f'SVENPY{mat:02d}'
    return gsw_dec[col] / 100.0

# --- Series 5: SVENY10 ZC 10yr ---
sveny10 = get_gsw_zc(10)
sveny09 = get_gsw_zc(9)
r_zc10 = zc_bond_return(sveny10, sveny09, 10)
results['ZC 10yr'] = dict(
    r_bond=r_zc10, xb=make_xb(r_zc10), D=None,
    yield_series=sveny10, method='ZC exact', n=10, label='ZC 10yr'
)

# --- Series 6: SVENY20 ZC 20yr ---
sveny20 = get_gsw_zc(20)
sveny19 = get_gsw_zc(19)
r_zc20 = zc_bond_return(sveny20, sveny19, 20)
results['ZC 20yr'] = dict(
    r_bond=r_zc20, xb=make_xb(r_zc20), D=None,
    yield_series=sveny20, method='ZC exact', n=20, label='ZC 20yr'
)

# --- Series 7: SVENY30 ZC 30yr ---
sveny30 = get_gsw_zc(30)
sveny29 = get_gsw_zc(29)
r_zc30 = zc_bond_return(sveny30, sveny29, 30)
results['ZC 30yr'] = dict(
    r_bond=r_zc30, xb=make_xb(r_zc30), D=None,
    yield_series=sveny30, method='ZC exact', n=30, label='ZC 30yr'
)

# --- Series 8: SVENPY10 par 10yr (CCV) ---
svenpy10 = get_gsw_par(10)
r_bond_svenpy10, D_svenpy10 = ccv_bond_return(svenpy10, 10)
results['Par 10yr'] = dict(
    r_bond=r_bond_svenpy10, xb=make_xb(r_bond_svenpy10), D=D_svenpy10,
    yield_series=svenpy10, method='CCV', n=10, label='Par 10yr'
)

# --- Series 9: SVENPY20 par 20yr (CCV) ---
svenpy20 = get_gsw_par(20)
r_bond_svenpy20, D_svenpy20 = ccv_bond_return(svenpy20, 20)
results['Par 20yr'] = dict(
    r_bond=r_bond_svenpy20, xb=make_xb(r_bond_svenpy20), D=D_svenpy20,
    yield_series=svenpy20, method='CCV', n=20, label='Par 20yr'
)

# --- Series 10: SVENPY30 par 30yr (CCV) ---
svenpy30 = get_gsw_par(30)
r_bond_svenpy30, D_svenpy30 = ccv_bond_return(svenpy30, 30)
results['Par 30yr'] = dict(
    r_bond=r_bond_svenpy30, xb=make_xb(r_bond_svenpy30), D=D_svenpy30,
    yield_series=svenpy30, method='CCV', n=30, label='Par 30yr'
)

print(f'Constructed {len(results)} series.')

## 5. Verification Checks

In [ ]:
print('=== CCV Duration Sanity ===')
for label, res in results.items():
    if res['method'] == 'CCV' and res['D'] is not None:
        D_mean = res['D'].dropna().mean()
        print(f"  {label:16s}: mean D = {D_mean:.2f}  (n={res['n']})")

print()
print('=== Return Identity: rtb + xb == r_bond - pi ===')
for label, res in results.items():
    r_bond = res['r_bond'].dropna()
    xb = res['xb'].dropna()
    common = r_bond.index.intersection(xb.index).intersection(rtb.dropna().index).intersection(pi.dropna().index)
    lhs = (rtb.reindex(common) + xb.reindex(common))
    rhs = (r_bond.reindex(common) - pi.reindex(common))
    resid = (lhs - rhs).abs().max()
    print(f"  {label:16s}: max |resid| = {resid:.2e}  {'PASS' if resid < 1e-10 else 'FAIL'}")

print()
print('=== Sign Check: Large yield changes ===')
check_years = {2008: 'yields fell', 2019: 'yields fell', 2020: 'yields fell', 2022: 'yields rose'}
for yr, desc in check_years.items():
    print(f'  Year {yr} ({desc}):')
    for label, res in results.items():
        xb = res['xb']
        if yr in xb.index and not np.isnan(xb.loc[yr]):
            sign = '+' if xb.loc[yr] > 0 else '-'
            ok = ('yields fell' in desc and xb.loc[yr] > 0) or ('yields rose' in desc and xb.loc[yr] < 0)
            print(f"    {label:16s}: xb={xb.loc[yr]*100:+6.1f}%  {'OK' if ok else 'CHECK'}")
    print()

In [ ]:
print('=== Cross-Check: CCV Par vs ZC at Same Maturity ===')
pairs = [
    ('Par 10yr', 'ZC 10yr', 10),
    ('Par 20yr', 'ZC 20yr', 20),
    ('Par 30yr', 'ZC 30yr', 30),
]
for par_label, zc_label, n in pairs:
    xb_par = results[par_label]['xb'].dropna()
    xb_zc = results[zc_label]['xb'].dropna()
    common = xb_par.index.intersection(xb_zc.index)
    if len(common) > 5:
        corr = xb_par.reindex(common).corr(xb_zc.reindex(common))
        print(f"  {n}yr:  corr(xb_par, xb_zc) = {corr:.4f}  (expect >0.95)")
        print(f"         E[xb_par] = {xb_par.reindex(common).mean()*100:.2f}%,  E[xb_zc] = {xb_zc.reindex(common).mean()*100:.2f}%")
        print(f"         std(xb_par) = {xb_par.reindex(common).std()*100:.2f}%,  std(xb_zc) = {xb_zc.reindex(common).std()*100:.2f}%")
    else:
        print(f"  {n}yr:  insufficient overlap ({len(common)} years)")
    print()

## 6. Comparison Table

In [ ]:
def compute_stats(res, sample_start=None, sample_end=None):
    """Compute all stats for a bond series on a given sample."""
    xb = res['xb'].dropna()
    r_bond = res['r_bond'].dropna()
    Y = res['yield_series'].dropna()
    
    # Restrict to sample
    if sample_start:
        xb = xb[xb.index >= sample_start]
        r_bond = r_bond[r_bond.index >= sample_start]
        Y = Y[Y.index >= sample_start]
    if sample_end:
        xb = xb[xb.index <= sample_end]
        r_bond = r_bond[r_bond.index <= sample_end]
        Y = Y[Y.index <= sample_end]
    
    common = xb.index.intersection(rtb.dropna().index).intersection(pi.dropna().index).intersection(xr.dropna().index)
    xb_c = xb.reindex(common)
    rtb_c = rtb.reindex(common)
    pi_c = pi.reindex(common)
    xr_c = xr.reindex(common)
    
    if len(common) < 3:
        return None
    
    # Total real log return = rtb + xb
    real_log = rtb_c + xb_c
    # Mean total real gross return - 1
    real_gross_mean = np.exp(real_log).mean() - 1
    
    # Duration
    if res['method'] == 'CCV' and res['D'] is not None:
        D_common = res['D'].reindex(common).dropna()
        eff_dur = D_common.mean()
    else:
        eff_dur = res['n']  # ZC: duration = maturity
    
    # Mean yield over sample
    Y_common = Y.reindex(common).dropna()
    
    return {
        'Years': f"{common.min()}–{common.max()}",
        'N': len(common),
        'E[xb] %': xb_c.mean() * 100,
        'E[rtb+xb] %': real_log.mean() * 100,
        'E[gross]-1 %': real_gross_mean * 100,
        'std(xb) %': xb_c.std() * 100,
        'Sharpe': xb_c.mean() / xb_c.std() if xb_c.std() > 0 else np.nan,
        'Mean yield %': Y_common.mean() * 100 if len(Y_common) > 0 else np.nan,
        'Eff. duration': eff_dur,
        'corr(xb,xr)': xb_c.corr(xr_c),
        'corr(xb,rtb)': xb_c.corr(rtb_c),
    }

print('Stats function defined.')

In [ ]:
# Full sample stats
print('=' * 100)
print('FULL SAMPLE (maximum available for each series)')
print('=' * 100)

rows = []
for label, res in results.items():
    stats = compute_stats(res)
    if stats:
        stats['Series'] = label
        stats['Method'] = res['method']
        stats['n'] = res['n']
        rows.append(stats)

df_full = pd.DataFrame(rows)
cols = ['Series', 'Method', 'n', 'Years', 'N', 'E[xb] %', 'E[rtb+xb] %', 'E[gross]-1 %',
        'std(xb) %', 'Sharpe', 'Mean yield %', 'Eff. duration', 'corr(xb,xr)', 'corr(xb,rtb)']
df_full = df_full[cols]
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')
print(df_full.to_string(index=False))
print()

In [ ]:
# Matched 1963-2025 sample
print('=' * 100)
print('MATCHED SAMPLE: 1963–2025')
print('=' * 100)

rows_matched = []
for label, res in results.items():
    stats = compute_stats(res, sample_start=1963, sample_end=2025)
    if stats and stats['N'] >= 10:
        stats['Series'] = label
        stats['Method'] = res['method']
        stats['n'] = res['n']
        rows_matched.append(stats)

df_matched = pd.DataFrame(rows_matched)
df_matched = df_matched[cols]
print(df_matched.to_string(index=False))
print()

## 7. Detailed Time Series Comparison

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Panel 1: Excess returns for CCV series
ax = axes[0]
for label in ['AAA 20yr', 'GS20 20yr', 'DGS30 30yr', 'DGS30 n=20']:
    xb = results[label]['xb'].dropna()
    ax.plot(xb.index, xb.values * 100, label=label, alpha=0.8)
ax.axhline(0, color='k', lw=0.5)
ax.set_ylabel('Excess return xb (%)')
ax.set_title('CCV Par/Coupon Bond Returns')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel 2: Excess returns for ZC series
ax = axes[1]
for label in ['ZC 10yr', 'ZC 20yr', 'ZC 30yr']:
    xb = results[label]['xb'].dropna()
    ax.plot(xb.index, xb.values * 100, label=label, alpha=0.8)
ax.axhline(0, color='k', lw=0.5)
ax.set_ylabel('Excess return xb (%)')
ax.set_title('Zero-Coupon Bond Returns (Higher Duration = More Volatile)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel 3: CCV par vs ZC at 20yr
ax = axes[2]
for label in ['AAA 20yr', 'Par 20yr', 'ZC 20yr']:
    xb = results[label]['xb'].dropna()
    ax.plot(xb.index, xb.values * 100, label=label, alpha=0.8)
ax.axhline(0, color='k', lw=0.5)
ax.set_ylabel('Excess return xb (%)')
ax.set_title('20yr Comparison: AAA vs Treasury Par vs Treasury ZC')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cumulative excess returns
fig, ax = plt.subplots(figsize=(14, 6))

for label in ['AAA 20yr', 'GS20 20yr', 'DGS30 30yr', 'ZC 10yr', 'ZC 20yr', 'Par 20yr']:
    xb = results[label]['xb'].dropna()
    cum = xb.cumsum()
    ax.plot(cum.index, cum.values * 100, label=label, alpha=0.8)

ax.set_ylabel('Cumulative excess return (%)')
ax.set_title('Cumulative Excess Bond Returns Over Bills')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Variance-Covariance Summary

Key for the VAR model: what enters as `xb` determines `Sigma_rr[xb,xb]` and cross-correlations.

In [ ]:
print('=== Variance-Covariance on 1963-2025 (where available) ===')
print()

for label, res in results.items():
    xb = res['xb'].dropna()
    xb = xb[(xb.index >= 1963) & (xb.index <= 2025)]
    common = xb.index.intersection(rtb.dropna().index).intersection(xr.dropna().index)
    if len(common) < 10:
        continue
    xb_c = xb.reindex(common)
    rtb_c = rtb.reindex(common)
    xr_c = xr.reindex(common)
    
    joint = pd.DataFrame({'rtb': rtb_c, 'xr': xr_c, 'xb': xb_c})
    cov = joint.cov()
    corr = joint.corr()
    
    print(f'--- {label} (N={len(common)}, {common.min()}-{common.max()}) ---')
    print(f'  Var(xb) = {cov.loc["xb","xb"]*1e4:.2f} (x1e-4),  std(xb) = {np.sqrt(cov.loc["xb","xb"])*100:.2f}%')
    print(f'  Cov(xb,xr) = {cov.loc["xb","xr"]*1e4:.2f},  Corr(xb,xr) = {corr.loc["xb","xr"]:.3f}')
    print(f'  Cov(xb,rtb) = {cov.loc["xb","rtb"]*1e4:.2f},  Corr(xb,rtb) = {corr.loc["xb","rtb"]:.3f}')
    print(f'  Cov(rtb,xr) = {cov.loc["rtb","xr"]*1e4:.2f}  (invariant to bond choice)')
    print()

## 9. Sorted Ranking by Sharpe and E[xb]

In [ ]:
if len(df_matched) > 0:
    print('=== Ranked by Sharpe Ratio (1963-2025 matched sample) ===')
    print(df_matched.sort_values('Sharpe', ascending=False)[['Series', 'Method', 'n', 'N', 'E[xb] %', 'std(xb) %', 'Sharpe', 'Eff. duration']].to_string(index=False))
    print()
    print('=== Ranked by E[xb] ===')
    print(df_matched.sort_values('E[xb] %', ascending=False)[['Series', 'Method', 'n', 'N', 'E[xb] %', 'std(xb) %', 'Sharpe', 'Eff. duration']].to_string(index=False))

if len(df_full) > 0:
    print()
    print('=== Ranked by Sharpe Ratio (full sample) ===')
    print(df_full.sort_values('Sharpe', ascending=False)[['Series', 'Method', 'n', 'N', 'E[xb] %', 'std(xb) %', 'Sharpe', 'Eff. duration']].to_string(index=False))

## 10. What Enters the VAR

Impact analysis: how does switching the bond series change the VAR moments?

In [ ]:
# Build the full 6-variable dataset for each candidate bond series
# that overlaps with 1963-2025 and uses the CCV method (suitable for the VAR)

# Shiller CAPE for cy
shiller['CAPE'] = pd.to_numeric(shiller['CAPE'], errors='coerce')
shiller_dec_cape = shiller[shiller.index.month == 12][['CAPE']].copy()
shiller_dec_cape.index = shiller_dec_cape.index.year
cy = -np.log(shiller_dec_cape['CAPE'])
cy.name = 'cy'

print('=== VAR sample statistics by bond choice (1963-2025) ===')
print()

# Current model baseline
spr_aaa = y_aaa - y_1
candidates = {
    'AAA 20yr (current)': ('AAA 20yr', y_aaa, spr_aaa),
    'DGS30 30yr': ('DGS30 30yr', y_dgs30, y_dgs30 - y_1),
    'Par 20yr (GSW)': ('Par 20yr', svenpy20 + y_1, svenpy20),  # spr = par20 - y_1, so y_20 = svenpy20 + y_1? No.
}

# Actually, spr = y_long - y_1 for each series. Let's just compare xb directly.
for label, res in results.items():
    xb = res['xb'].dropna()
    xb_63 = xb[(xb.index >= 1963) & (xb.index <= 2025)]
    if len(xb_63) < 20:
        continue
    print(f'{label:16s}:  E[xb] = {xb_63.mean()*100:+.2f}%,  std(xb) = {xb_63.std()*100:.2f}%,  '
          f'var(xb) = {xb_63.var()*1e4:.2f} (x1e-4),  N={len(xb_63)}')